# CEPU China Mainland - Exploratory Data Analysis

This notebook explores the China Economic Policy Uncertainty (CEPU) dataset from mainland newspapers.

**Dataset:** `cepu-china-mainland-paper.xlsx`

**Description:** Monthly Economic Policy Uncertainty index for China from 1949 onwards.

**Source:** Economic Policy Uncertainty in China Since 1949: The View from Mainland Newspapers, by Steven J. Davis, Dingqian Liu and Xuguang S. Sheng, 2019.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_PATH = Path('../../datasets/raw/cepu-china-mainland-paper.xlsx')
CSV_OUTPUT_PATH = Path('../../datasets/raw/cepu-china-mainland-paper.csv')

## 1. Load and Clean Data

In [ ]:
# Load Excel file
df_raw = pd.read_excel(DATA_PATH)

print("Raw data (first 10 rows):")
display(df_raw.head(10))

# Keep only the first 3 columns (year, month, EPU)
df = df_raw.iloc[:, :3].copy()
df.columns = ['year', 'month', 'EPU']

# Remove rows with missing EPU values
df = df.dropna(subset=['EPU'])

# Convert to appropriate types
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
df['EPU'] = df['EPU'].astype(float)

# Create date column
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

print("\nCleaned dataset:")
display(df.head())
print("\nData types:")
print(df.dtypes)

## 2. Save to CSV

In [ ]:
# Save cleaned data to CSV
df.to_csv(CSV_OUTPUT_PATH, index=False)
print(f"✓ Data saved to: {CSV_OUTPUT_PATH}")
print(f"  Rows saved: {len(df)}")

## 3. Data Overview

In [ ]:
print("Dataset Shape:", df.shape)
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("Total months:", len(df))
print("\nDataset Info:")
df.info()

## 4. Descriptive Statistics

In [ ]:
print("Descriptive Statistics for EPU Index:")
display(df['EPU'].describe())

print("\nAdditional Statistics:")
print(f"Skewness: {df['EPU'].skew():.4f}")
print(f"Kurtosis: {df['EPU'].kurtosis():.4f}")

## 5. Time Series Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['date'], df['EPU'], linewidth=1, alpha=0.8)
ax.set_title('China Economic Policy Uncertainty Index (1949-Present)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('EPU Index', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# With moving average
df['MA_12m'] = df['EPU'].rolling(window=12, center=True).mean()
df['MA_60m'] = df['EPU'].rolling(window=60, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['date'], df['EPU'], linewidth=0.5, alpha=0.4, label='Monthly EPU')
ax.plot(df['date'], df['MA_12m'], linewidth=1.5, label='12-month MA', color='orange')
ax.plot(df['date'], df['MA_60m'], linewidth=2, label='60-month MA', color='red')
ax.set_title('China EPU Index with Moving Averages', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('EPU Index', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['EPU'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(df['EPU'].mean(), color='red', linestyle='--', label=f'Mean: {df["EPU"].mean():.2f}')
axes[0].axvline(df['EPU'].median(), color='green', linestyle='--', label=f'Median: {df["EPU"].median():.2f}')
axes[0].set_title('Distribution of EPU Index', fontsize=12, fontweight='bold')
axes[0].set_xlabel('EPU Index')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].boxplot(df['EPU'], vert=True)
axes[1].set_title('Box Plot of EPU Index', fontsize=12, fontweight='bold')
axes[1].set_ylabel('EPU Index')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Decadal Analysis

In [ ]:
# Create decade column
df['decade'] = (df['year'] // 10) * 10

# Decade statistics
decade_stats = df.groupby('decade')['EPU'].agg(['mean', 'std', 'min', 'max', 'count'])
print("Decade Statistics:")
display(decade_stats)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(decade_stats.index, decade_stats['mean'], alpha=0.7, edgecolor='black', width=8)
ax.set_title('Average EPU Index by Decade', fontsize=14, fontweight='bold')
ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Average EPU Index', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Key Events

In [ ]:
# Highest uncertainty periods
top_20 = df.nlargest(20, 'EPU')[['date', 'year', 'month', 'EPU']]
print("Top 20 Highest Uncertainty Periods:")
display(top_20)

## 9. Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS - CHINA EPU (MAINLAND NEWSPAPERS)")
print("=" * 60)
print(f"\n1. Dataset Coverage:")
print(f"   - Start: {df['date'].min().strftime('%Y-%m')}")
print(f"   - End: {df['date'].max().strftime('%Y-%m')}")
print(f"   - Total Months: {len(df)}")
print(f"   - Years Covered: {df['date'].max().year - df['date'].min().year + 1}")
print(f"\n2. EPU Index Statistics:")
print(f"   - Mean: {df['EPU'].mean():.2f}")
print(f"   - Median: {df['EPU'].median():.2f}")
print(f"   - Std Dev: {df['EPU'].std():.2f}")
print(f"   - Min: {df['EPU'].min():.2f} ({df.loc[df['EPU'].idxmin(), 'date'].strftime('%Y-%m')})")
print(f"   - Max: {df['EPU'].max():.2f} ({df.loc[df['EPU'].idxmax(), 'date'].strftime('%Y-%m')})")
print(f"\n3. Temporal Insights:")
print(f"   - Decade with highest avg: {decade_stats['mean'].idxmax()}s ({decade_stats['mean'].max():.2f})")
print(f"   - Decade with lowest avg: {decade_stats['mean'].idxmin()}s ({decade_stats['mean'].min():.2f})")
print(f"\n4. Data Quality:")
print(f"   - Missing Values: {df.isnull().sum().sum()}")
print(f"   - CSV file saved: {CSV_OUTPUT_PATH}")
print("\n" + "=" * 60)